In [ ]:
import sys
import os

current_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(current_dir, '..')))

from src.parser import PDFParser
from src.chunking import PDFChunker
from src.embedding_gemini import GeminiEmbedder
from src.inferencer import GeminiInferencer
from src.vector_store import get_qdrant_vector_store


1. Parsing PDFs:

In [5]:
documents = PDFParser("../data/raw/BOE-A-2024-2248.pdf").parse()
print("Número de documentos:", len(documents))
print("\n--- Page content of the first document ---")
print(documents[0].page_content[:500])
print("\n\n--- Metadata of the first document ---")
print(documents[0].metadata)

Número de documentos: 13

--- Page content of the first document ---
I. DISPOSICIONES GENERALES
MINISTERIO DE LA PRESIDENCIA, JUSTICIA
Y RELACIONES CON LAS CORTES
2248 Real Decreto 141/2024, de 6 de febrero, por el que se modifica el 
Reglamento de Población y Demarcación Territorial de las Entidades Locales, 
aprobado por el Real Decreto 1690/1986, de 11 de julio.
En el marco de la transformación digital y modernización de las entidades locales una de 
las necesidades identificadas es la modernización de la gestión del padrón municipal.
El padrón municipal es un


--- Metadata of the first document ---
{'producer': 'Antenna House PDF Output Library 6.6.1477 (Linux64)', 'creator': 'eBOE', 'creationdate': '2024-02-06T18:14:01+01:00', 'keywords': 'DECRETO 141/2024 de 06/02/2024;"MINISTERIO DE LA PRESIDENCIA, JUSTICIA Y RELACIONES CON LAS CORTES";BOE-A-2024-2248;BOE 33 de 2024;2248;07/02/2024', 'moddate': '2024-02-06T18:38:07+01:00', 'trapped': '/False', 'subject': 'BOE-A-2024-2248', 'aut

2. Chunking:

In [6]:
chunker = PDFChunker(chunk_size=1000, chunk_overlap=100)
documents = chunker.chunk(documents)
print("\nNúmero de chunks:", len(documents))
print("\n--- Page content of the first chunk ---")
print(documents[0].page_content)
print("\n\n--- Metadata of the first chunk ---")
print(documents[0].metadata)


Número de chunks: 54

--- Page content of the first chunk ---
I. DISPOSICIONES GENERALES
MINISTERIO DE LA PRESIDENCIA, JUSTICIA
Y RELACIONES CON LAS CORTES
2248 Real Decreto 141/2024, de 6 de febrero, por el que se modifica el 
Reglamento de Población y Demarcación Territorial de las Entidades Locales, 
aprobado por el Real Decreto 1690/1986, de 11 de julio.
En el marco de la transformación digital y modernización de las entidades locales una de 
las necesidades identificadas es la modernización de la gestión del padrón municipal.
El padrón municipal es un registro administrativo que reviste una gran importancia, 
por cuanto la inscripción en el mismo supone el requisito previo e imprescindible para 
que las personas puedan tener acceso, con todas las garantías, al uso y disfrute de los 
servicios públicos de educación, sanidad y servicios sociales en un municipio, y al 
ejercicio del derecho de sufragio, entre otros aspectos.
Debido precisamente a la relevancia que este registro oste

In [7]:
# Embedding & Vector Store (Langchain Qdrant en memoria)
embedder = GeminiEmbedder()

# Obtenemos el wrapper the LangChain ya configurado
vectorstore = get_qdrant_vector_store(embedder, collection_name="rag_test")

print("Ingestando documentos en Qdrant (LangChain)...")
vectorstore.add_documents(documents)
print("¡Documentos ingestados con éxito!")

Ingestando documentos en Qdrant (LangChain)...
¡Documentos ingestados con éxito!


In [9]:
vectorstore

In [12]:
# check vector store number of documents

In [ ]:
llm = GeminiInferencer()

# Simulación de prompt
pregunta = "¿Cuál es el resumen principal de este documento?"

# Recuperación de documentos relevantes a través del API de Langchain (similarity_search)
resultados_busqueda = vectorstore.similarity_search(pregunta, k=3)
contexto = "\n\n".join([doc.page_content for doc in resultados_busqueda])

# Generación
prompt = f"""Responde a la pregunta basándote únicamente en el contexto proporcionado.
Contexto: {contexto}

Pregunta: {pregunta}"""

respuesta = llm.infer(prompt)

print(f"Respuesta del RAG: {respuesta}")


Respuesta del RAG: Respuesta: El documento detalla la reforma del Reglamento de Población y Demarcación Territorial de las Entidades Locales (y de la Ley 7/1985), enmarcada en el Hito 147 del Componente 11 del PRTR. Su objetivo principal es modernizar y digitalizar la gestión del padrón municipal, permitiendo la interconexión en tiempo real entre los ayuntamientos y el Instituto Nacional de Estadística (INE). Esta modificación busca adaptar la normativa a los cambios legales recientes, garantizando la seguridad jurídica, la transparencia y la eficiencia en la gestión pública sin imponer nuevas cargas administrativas a la ciudadanía o a las empresas.
